In [1]:
import ee
import math
from datetime import datetime, timedelta
import requests
from PIL import Image, ImageDraw
from io import BytesIO
import time
ee.Initialize()

/home/wmlegion/miniconda3/envs/agri_land_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
import ee
import requests
from datetime import datetime
from pathlib import Path
from io import BytesIO
from PIL import Image

from tile_utils import draw_marker


# Reemplaza con tu project ID de Google Cloud vinculado a Earth Engine
#GEE_PROJECT = "tu-project-id-aqui"


def get_point_sentinel2(lat, lon, buffer_m=1000, start_date="2025-01-01", end_date=None,
                         max_cloud_pct=20, out_prefix="point_sentinel2",
                         output_dir="../img/maps", show_marker=True, dimensions=512):
    """
    lat, lon: coordenadas del punto central
    buffer_m: radio en metros alrededor del punto (define el tamaño del recorte)
    start_date, end_date: rango de fechas donde buscar imágenes (end_date=None -> hoy)
    max_cloud_pct: nubosidad máxima aceptada (%). Sube este valor si no encuentra
                   ninguna imagen en zonas muy nubladas.
    dimensions: tamaño en pixeles del lado más largo de la imagen descargada
    """
#    ee.Initialize(project=GEE_PROJECT)

    end_date = end_date or datetime.now().strftime("%Y-%m-%d")

    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(buffer_m).bounds()

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud_pct))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )

    n_images = collection.size().getInfo()
    if n_images == 0:
        raise ValueError(
            f"No se encontraron imágenes con menos de {max_cloud_pct}% de nubes "
            f"entre {start_date} y {end_date}. Prueba subiendo max_cloud_pct o "
            f"ampliando el rango de fechas."
        )

    image = collection.first()
    info = image.getInfo()
    acquisition_date = datetime.utcfromtimestamp(
        info["properties"]["system:time_start"] / 1000
    ).strftime("%Y-%m-%d")
    cloud_pct = info["properties"]["CLOUDY_PIXEL_PERCENTAGE"]
    print(f"[DEBUG] {n_images} imagenes candidatas encontradas")
    print(f"[DEBUG] Imagen seleccionada: {acquisition_date}, nubosidad: {cloud_pct:.1f}%")

    # B4=rojo, B3=verde, B2=azul -> color real. max=3000 es un estiramiento
    # típico para reflectancia de superficie de Sentinel-2 (escala 0-10000).
    vis_image = image.select(["B4", "B3", "B2"]).visualize(min=0, max=3000)

    thumb_url = vis_image.getThumbURL({
        "region": region,
        "dimensions": dimensions,
        "format": "png",
    })

    resp = requests.get(thumb_url, timeout=30)
    resp.raise_for_status()
    canvas = Image.open(BytesIO(resp.content)).convert("RGB")

    if show_marker:
        # Aproximación lineal dentro del bbox (válida para buffers chicos como este)
        coords = region.bounds().getInfo()["coordinates"][0]
        lons = [c[0] for c in coords]
        lats_ = [c[1] for c in coords]
        west, east = min(lons), max(lons)
        south, north = min(lats_), max(lats_)

        px = (lon - west) / (east - west) * canvas.width
        py = (north - lat) / (north - south) * canvas.height
        draw_marker(canvas, px, py)

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename
    canvas.save(out_path)

    print(f"Imagen guardada en {out_path} ({canvas.width}x{canvas.height}px)")
    return out_path


if __name__ == "__main__":
    get_point_sentinel2(7.3297, -73.1867, buffer_m=1000, max_cloud_pct=20)

[DEBUG] 16 imagenes candidatas encontradas
[DEBUG] Imagen seleccionada: 2025-01-22, nubosidad: 1.5%
Imagen guardada en ../img/maps/point_sentinel2-v260804194212.png (512x512px)


In [ ]:
## MDS650_v260805_regional_elevation_map

from datetime import datetime
from io import BytesIO
from pathlib import Path
import requests
import ee
from PIL import Image

from tile_utils import draw_marker


# Paleta topográfica: valles/tierras bajas en verde, zonas medias en
# amarillo/marrón, cumbres en blanco/gris (referencia orientativa de rangos
# altitudinales para Colombia, ajustar 'max' si tu región es más alta/baja)
ELEVATION_PALETTE = [
    '004400',  # 0 – 364 m: Valles profundos, tierras bajas o planicies costeras
    '006600',  # 364 – 727 m: Tierras bajas / Bosques húmedos tropicales
    '38a800',  # 727 – 1,091 m: Transición baja / Inicio de piedemontes
    '73d216',  # 1,091 – 1,455 m: Zonas cafeteras o agrícolas de clima templado
    'b2d235',  # 1,455 – 1,818 m: Laderas de montaña media
    'fce94f',  # 1,818 – 2,182 m: Tierras medias altas / Bosques andinos
    'e9b96e',  # 2,182 – 2,545 m: Zonas altoandinas / Transición fría
    'c87d32',  # 2,545 – 2,909 m: Páramos bajos / Montaña alta
    '8f5902',  # 2,909 – 3,273 m: Páramos altos / Suelos rocosos fríos
    '5c3a21',  # 3,273 – 3,636 m: Superpáramo / Cumbres escarpadas
    '8b8b8b',  # 3,636 – 4,000+ m: Picos más altos / Gris roca alpino
]


def download_regional_elevation_map(lat, lon, region_type="state", out_prefix="regional_elevation",
                                     output_dir="../img/maps", dimensions=720,
                                     min_elevation=0, max_elevation=4000, show_marker=True):
    """
    Descarga un mapa de elevación (DEM, paleta topográfica) de la región
    administrativa (state o country) que contiene el punto dado.

    lat, lon: coordenadas del punto de referencia (define la región Y se
               marca en el mapa si show_marker=True)
    region_type: "state" (nivel 1, departamento/provincia) o "country" (nivel 0)
    dimensions: tamaño en pixeles del lado más largo de la imagen
    min_elevation, max_elevation: rango de la paleta de colores (metros)
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    """
    point = ee.Geometry.Point([lon, lat])

    if region_type == "country":
        feature = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).first()
        name_key = 'ADM0_NAME'
    else:
        feature = ee.FeatureCollection("FAO/GAUL/2015/level1").filterBounds(point).first()
        name_key = 'ADM1_NAME'

    region_name = feature.get(name_key).getInfo()
    geom = feature.geometry()
    print(f"[DEBUG] Zona detectada: {region_name}")

    image = ee.Image('USGS/SRTMGL1_003').select('elevation')
    clipped = image.clip(geom)

    vis_image = clipped.visualize(
        min=min_elevation,
        max=max_elevation,
        palette=ELEVATION_PALETTE,
    )

    thumbnail_url = vis_image.getThumbURL({
        'region': geom,
        'dimensions': dimensions,
        'format': 'png',
    })

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    resp = requests.get(thumbnail_url, timeout=30)
    resp.raise_for_status()
    canvas = Image.open(BytesIO(resp.content)).convert("RGBA")

    if show_marker:
        # Interpolación lineal dentro del bbox de la geometría (aproximación
        # razonable para referencia visual; a escala de state/país hay algo
        # de distorsión por la proyección, pero sirve para ubicar el punto).
        bounds_coords = geom.bounds().getInfo()["coordinates"][0]
        lons = [c[0] for c in bounds_coords]
        lats = [c[1] for c in bounds_coords]
        west, east = min(lons), max(lons)
        south, north = min(lats), max(lats)

        px = (lon - west) / (east - west) * canvas.width
        py = (north - lat) / (north - south) * canvas.height
        draw_marker(canvas, px, py)

    canvas.save(out_path)
    print(f"Imagen guardada en {out_path} ({canvas.width}x{canvas.height}px)")
    return out_path


if __name__ == "__main__":
    
# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
# Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
    
    lat_test = 7.4584221918243045
    lon_test = -73.222052853104

    download_regional_elevation_map(lat_test, lon_test, region_type="state")